# 🚀 Bane Agent: Neural DPO Benchmark on Kaggle GPU (16GB T4)

This notebook runs our fine-tuned DPO LLaMA-3 model (`bane_dpo_lora_adapters`) with **full hardware acceleration** on an NVIDIA T4 GPU.
It benchmarks all **30 Enterprise Queries** (20 Technical + 10 Conversational Slang) against `enterprise_nexus.sqlite`.

### Step 1: Install High-Performance GPU Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets
!pip install faiss-cpu sentence-transformers rich fastapi uvicorn requests

### Step 2: Clone the Bane Agent Repository & Pull Code

In [ ]:
import os
import sys

if not os.path.exists("/kaggle/working/Bane_Agent") and not os.path.exists("Bane_Agent"):
    !git clone https://github.com/pariveshkoshta-spec/Bane_Agent.git

if os.path.exists("/kaggle/working/Bane_Agent"):
    %cd /kaggle/working/Bane_Agent
elif os.path.exists("Bane_Agent"):
    %cd Bane_Agent

!git pull

### Step 3: Verify GPU Acceleration & LoRA Adapter Presence

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Please turn on GPU acceleration in Kaggle: Settings -> Accelerator -> GPU T4 x2")

# Check adapter presence
candidates = [
    "/kaggle/working/bane_dpo_lora_adapters",
    "/kaggle/working/outputs/checkpoint-375",
    "results/bane_dpo_lora_adapters",
    "/kaggle/input/bane-dpo-lora-adapters"
]
for c in candidates:
    if os.path.exists(c):
        print(f"Found adapters at: {c}")

### Step 4: Run the Full 30-Question GPU Benchmark!
This runs our fine-tuned model against `enterprise_nexus.sqlite`, executes every generated query on the database, and measures speed and accuracy.

In [ ]:
!python scripts/run_gpu_benchmark.py

### Step 5: View the Generated Benchmark Report & Gap Analysis

In [ ]:
from IPython.display import display, Markdown
if os.path.exists("KAGGLE_GPU_BENCHMARK_REPORT.md"):
    with open("KAGGLE_GPU_BENCHMARK_REPORT.md", "r") as f:
        report_content = f.read()
    display(Markdown(report_content[:3000] + "\n\n*(truncated for preview)*"))
else:
    print("Report not generated yet.")

### Bonus: Expose Cloud GPU API via Public Tunnel
Run this cell if you want your local Mac CLI (`bane query "..."`) to talk directly to this Kaggle GPU!

In [ ]:
# Install pyngrok or cloudflared
!pip install pyngrok
from pyngrok import ngrok
import subprocess
import time

# Start uvicorn in background
proc = subprocess.Popen(["uvicorn", "src.api:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

# Expose port 8000 via ngrok tunnel (replace with your ngrok token if needed)
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")
# public_url = ngrok.connect(8000)
# print(f"\n🚀 LIVE GPU API URL: {public_url}")
# print(f"On your Mac, run: export BANE_API_URL={public_url}")